## 使用嵌入

本 notebook 包含一些实用代码片段，可用于通过 OpenAI API 调用 `text-embedding-3-small` 模型对文本进行嵌入。

In [ ]:
from openai import OpenAI
client = OpenAI()

embedding = client.embeddings.create(
    input="Your text goes here", model="text-embedding-3-small"
).data[0].embedding
len(embedding)


1536

建议使用 'tenacity' 包或其他指数退避（exponential backoff）实现来更好地应对 API 速率限制，因为过快、过多地调用 API 可能会触发速率限制。使用下面的函数可以确保你尽快获得嵌入结果。

In [ ]:
# Negative example (slow and rate-limited)
from openai import OpenAI
client = OpenAI()

num_embeddings = 10000 # Some large number
for i in range(num_embeddings):
    embedding = client.embeddings.create(
        input="Your text goes here", model="text-embedding-3-small"
    ).data[0].embedding
    print(len(embedding))

In [2]:
# Best practice
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import OpenAI
client = OpenAI()

# Retry up to 6 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_embedding(text: str, model="text-embedding-3-small") -> list[float]:
    return client.embeddings.create(input=[text], model=model).data[0].embedding

embedding = get_embedding("Your text goes here", model="text-embedding-3-small")
print(len(embedding))

1536
